# Handwritten Digit Recognition (MNIST)

**Goal:** Train models to classify handwritten digits (0-9) from images, comparing a classical ML baseline against a CNN.

**Pipeline:**
1. Load & explore the MNIST dataset
2. Train classical baselines (SVM, MLP)
3. Train a small CNN
4. Evaluate all models with accuracy / F1
5. Visualize predictions and errors

## 1. Setup & imports

In [ ]:
import sys
sys.path.append('../src')

import numpy as np
import matplotlib.pyplot as plt

from data_loader import load_raw_mnist, load_for_classical_ml, load_for_cnn
from visualize import plot_sample_digits, plot_predictions, plot_confusion_matrix, plot_training_history

from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report

from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping

## 2. Load & explore the data

In [ ]:
(x_train, y_train), (x_test, y_test) = load_raw_mnist()
print('Train:', x_train.shape, y_train.shape)
print('Test: ', x_test.shape, y_test.shape)
print('Classes:', np.unique(y_train))

In [ ]:
# Look at a few sample digits
plot_sample_digits(x_train, y_train, n=10, title='Sample training digits')

In [ ]:
# Class balance check
unique, counts = np.unique(y_train, return_counts=True)
plt.bar(unique, counts)
plt.xlabel('Digit')
plt.ylabel('Count')
plt.title('Training set class distribution')
plt.show()

## 3. Baseline 1: Support Vector Machine (SVM)

SVMs are slow on large datasets, so we subsample the training set to ~8,000 images to keep training time reasonable.

In [ ]:
X_train_svm, y_train_svm, X_test_flat, y_test_flat = load_for_classical_ml(sample_size=8000)

svm_model = SVC(kernel='rbf', C=5, gamma='scale')
svm_model.fit(X_train_svm, y_train_svm)

svm_pred = svm_model.predict(X_test_flat)
svm_acc = accuracy_score(y_test_flat, svm_pred)
svm_f1 = f1_score(y_test_flat, svm_pred, average='macro')
print(f'SVM Accuracy: {svm_acc:.4f} | Macro F1: {svm_f1:.4f}')

## 4. Baseline 2: Multi-Layer Perceptron (MLP)

In [ ]:
X_train_full, y_train_full, X_test_full, y_test_full = load_for_classical_ml()

mlp_model = MLPClassifier(
    hidden_layer_sizes=(256, 128),
    activation='relu',
    solver='adam',
    max_iter=30,
    random_state=42,
    early_stopping=True,
)
mlp_model.fit(X_train_full, y_train_full)

mlp_pred = mlp_model.predict(X_test_full)
mlp_acc = accuracy_score(y_test_full, mlp_pred)
mlp_f1 = f1_score(y_test_full, mlp_pred, average='macro')
print(f'MLP Accuracy: {mlp_acc:.4f} | Macro F1: {mlp_f1:.4f}')

## 5. CNN Model

Now let's see how much a convolutional architecture improves things.

In [ ]:
X_train_cnn, y_train_cnn, X_test_cnn, y_test_cnn = load_for_cnn()

cnn_model = models.Sequential([
    layers.Input(shape=(28, 28, 1)),
    layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(10, activation='softmax'),
])

cnn_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
cnn_model.summary()

In [ ]:
early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

history = cnn_model.fit(
    X_train_cnn, y_train_cnn,
    validation_split=0.1,
    epochs=15,
    batch_size=128,
    callbacks=[early_stop],
    verbose=2,
)

In [ ]:
plot_training_history(history)

In [ ]:
cnn_pred_probs = cnn_model.predict(X_test_cnn)
cnn_pred = np.argmax(cnn_pred_probs, axis=1)
cnn_true = np.argmax(y_test_cnn, axis=1)

cnn_acc = accuracy_score(cnn_true, cnn_pred)
cnn_f1 = f1_score(cnn_true, cnn_pred, average='macro')
print(f'CNN Accuracy: {cnn_acc:.4f} | Macro F1: {cnn_f1:.4f}')
print(classification_report(cnn_true, cnn_pred, digits=3))

## 6. Compare all models

In [ ]:
results = {
    'SVM': (svm_acc, svm_f1),
    'MLP': (mlp_acc, mlp_f1),
    'CNN': (cnn_acc, cnn_f1),
}

print(f"{'Model':<8}{'Accuracy':<12}{'Macro F1':<12}")
for name, (acc, f1) in results.items():
    print(f"{name:<8}{acc:<12.4f}{f1:<12.4f}")

names = list(results.keys())
accs = [results[n][0] for n in names]
plt.bar(names, accs, color=['#4C72B0', '#DD8452', '#55A868'])
plt.ylabel('Test Accuracy')
plt.ylim(0.8, 1.0)
plt.title('Model comparison')
plt.show()

## 7. Visualize predictions & errors

In [ ]:
# Correct + incorrect CNN predictions, side by side
plot_predictions(x_test[:10], cnn_true[:10], cnn_pred[:10], n=10, title='CNN predictions (first 10 test images)')

In [ ]:
# Find and show only the CNN's mistakes
wrong_idx = np.where(cnn_pred != cnn_true)[0]
print(f'CNN got {len(wrong_idx)} out of {len(cnn_true)} test images wrong ({len(wrong_idx)/len(cnn_true)*100:.2f}%)')

if len(wrong_idx) > 0:
    sample_wrong = wrong_idx[:10]
    plot_predictions(x_test[sample_wrong], cnn_true[sample_wrong], cnn_pred[sample_wrong], n=len(sample_wrong), title='CNN mistakes')

In [ ]:
plot_confusion_matrix(cnn_true, cnn_pred, title='CNN Confusion Matrix')

## 8. Save the best model

In [ ]:
import joblib

cnn_model.save('../models/cnn_model.h5')
joblib.dump(mlp_model, '../models/mlp_model.pkl')
joblib.dump(svm_model, '../models/svm_model.pkl')
print('Models saved to ../models/')

## Summary

- The CNN outperforms both classical baselines because convolutional layers learn spatial patterns (edges, curves) instead of treating each pixel independently.
- The MLP performs reasonably well since MNIST digits are fairly simple, low-resolution images.
- The SVM, trained on a subsample, is competitive but training time scales poorly with more data.
- Most errors happen on ambiguous or messily-written digits (e.g. 4 vs 9, 3 vs 5) — visible in the confusion matrix above.